# Exercise 3 — fetch_ohlcv and normalize_ohlcv

`fetch_ohlcv` is the injectable wrapper around yfinance. Passing `fetch_fn` replaces the real API call — essential for gate testing. `normalize_ohlcv` strips any timezone information and keeps only the five standard columns, returning a clean copy.

In [ ]:
import pandas as pd, math, sqlite3, tempfile, os

OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

def _synthetic(ticker="TEST", period="1y", interval="1d", n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def validate_ohlcv(df):
    if not isinstance(df, pd.DataFrame): return False, "not a DataFrame"
    missing = [c for c in OHLCV_COLS if c not in df.columns]
    if missing: return False, "missing columns: " + ", ".join(missing)
    if len(df) == 0: return False, "DataFrame is empty"
    valid = df.dropna(subset=["High", "Low"])
    if len(valid) > 0 and (valid["High"] < valid["Low"]).any():
        return False, "High < Low detected"
    return True, ""

# ── Exercise: implement fetch_ohlcv and normalize_ohlcv ──────────────────────

def fetch_ohlcv(ticker, period="1y", interval="1d", fetch_fn=None):
    """Fetch OHLCV data for ticker.

    Args:
        ticker   : stock symbol, e.g. "AAPL"
        period   : data range, e.g. "1y", "6mo"
        interval : bar size, e.g. "1d", "1h"
        fetch_fn : callable(ticker, period, interval) -> DataFrame, or None.
                   None -> calls yfinance.

    Returns:
        pd.DataFrame with columns Open, High, Low, Close, Volume
    """
    # TODO:
    # 1. if fetch_fn is not None: return fetch_fn(ticker, period, interval)
    # 2. import yfinance as yf
    # 3. df = yf.download(ticker, period=period, interval=interval, progress=False)
    # 4. handle MultiIndex columns: if isinstance(df.columns, pd.MultiIndex):
    #        df.columns = df.columns.get_level_values(0)
    # 5. return df
    if fetch_fn is not None:
        return fetch_fn(ticker, period, interval)
    return _synthetic(ticker, period, interval)  # stub: removes when yfinance added


def normalize_ohlcv(df):
    """Return a copy of df with a tz-naive DatetimeIndex and only OHLCV_COLS.

    Steps:
      1. df = df.copy()
      2. If index is not DatetimeIndex: convert with pd.to_datetime
      3. If index has timezone info: strip it with tz_localize(None)
      4. Keep only columns that appear in OHLCV_COLS (in order)
    """
    # TODO: implement the four steps above
    return df.copy()


### Checks

In [ ]:
checks = 0

# 1 — fetch_ohlcv with fetch_fn injection returns the injected result
try:
    df = fetch_ohlcv("TEST", fetch_fn=_synthetic)
    assert isinstance(df, pd.DataFrame)
    assert "Close" in df.columns
    assert len(df) == 50
    checks += 1; print("✅ 1 fetch_fn injection returns injected result")
except Exception as e:
    print("❌ 1:", e)

# 2 — fetch_ohlcv passes ticker/period/interval to fetch_fn correctly
try:
    calls = []
    def _capturing(ticker, period, interval):
        calls.append((ticker, period, interval))
        return _synthetic()
    fetch_ohlcv("AAPL", period="6mo", interval="1d", fetch_fn=_capturing)
    assert calls == [("AAPL", "6mo", "1d")], f"unexpected calls: {calls}"
    checks += 1; print("✅ 2 fetch_fn receives correct ticker, period, interval")
except Exception as e:
    print("❌ 2:", e)

# 3 — normalize_ohlcv strips timezone from DatetimeIndex
try:
    df = _synthetic()
    df_tz = df.copy()
    df_tz.index = df_tz.index.tz_localize("UTC")
    normed = normalize_ohlcv(df_tz)
    assert normed.index.tz is None, f"timezone not stripped: {normed.index.tz}"
    checks += 1; print("✅ 3 normalize_ohlcv strips timezone")
except Exception as e:
    print("❌ 3:", e)

# 4 — normalize_ohlcv keeps only the 5 OHLCV_COLS
try:
    df = _synthetic()
    df["ExtraCol"] = 999
    normed = normalize_ohlcv(df)
    assert list(normed.columns) == OHLCV_COLS, f"columns: {list(normed.columns)}"
    checks += 1; print("✅ 4 normalize_ohlcv keeps only OHLCV_COLS in order")
except Exception as e:
    print("❌ 4:", e)

# 5 — normalize_ohlcv returns a copy (does not mutate input)
try:
    df = _synthetic()
    df_tz = df.copy()
    df_tz.index = df_tz.index.tz_localize("UTC")
    _ = normalize_ohlcv(df_tz)
    assert df_tz.index.tz is not None, "input was mutated!"
    checks += 1; print("✅ 5 normalize_ohlcv does not mutate the input DataFrame")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
